### Librerias

In [81]:
import kfp
from google.cloud import aiplatform
from kfp.v2 import dsl, compiler
from kfp.v2.dsl import (Artifact, ClassificationMetrics, Input, Metrics, Output, component, Dataset)
from google.cloud import storage
from google_cloud_pipeline_components.v1.vertex_notification_email import VertexNotificationEmailOp
from typing import NamedTuple

### Componente de Validacion

In [82]:
@component(packages_to_install=["google-cloud-bigquery"])
def validate_data(
    source_x_train_table: str,
) -> NamedTuple("Outputs", [("condition", str)], ):

    from google.cloud import bigquery

    client = bigquery.Client()

    try:
        client.get_table(source_x_train_table)
        condition = "true"

    except Exception as e:
        condition = "false"

    return (condition,)

### Componente de Captura de Error

In [83]:
@component
def error_op(msg: str):
    return msg

### Componente del Algoritmo de Clasificacion

In [84]:
@component(
    packages_to_install=[
        "google-cloud-bigquery",
        "google-cloud-bigquery-storage",
        "pandas",
        "scikit-learn",
        "xgboost",
        "joblib",
        "db-dtypes",
        "pyarrow",
        "pandas-gbq",
        "google-cloud-storage",
        "pytz"
    ],
)
def classify(
    project: str,
    source_x_train_table: str,
    table_id: str,
    model_path: str,
    encoder_path: str
):  
    import sys
    from datetime import datetime
    import pandas as pd
    from google.cloud import bigquery
    from google.auth import default
    import pandas_gbq
    from google.cloud import storage
    from joblib import load
    from io import BytesIO
    from pytz import timezone
    
    TZ = timezone('America/Lima')
    FORMAT_DATE = "%Y-%m-%d"
    
    client = bigquery.Client(project=project)
    
    Datos = client.query(
    '''SELECT * FROM `{dsource_table}`
        '''.format(dsource_table=source_x_train_table)
    ).to_dataframe()
    
    print('=================== Test Data =====================')
    print(Datos.head(5))
    
    def generate_datetime_created():
        return datetime.now(TZ).replace(tzinfo=None)
    
    def generate_date_created():
        return datetime.now(TZ).date().strftime(FORMAT_DATE)
    
    
    def load_artifact_from_gcs(path):
        storage_client = storage.Client()

        bucket_name, blob_name = path.replace("gs://", "").split("/", 1)

        bucket = storage_client.bucket(bucket_name)
        blob = bucket.blob(blob_name)
        model_bytes = blob.download_as_string()

        artifact = load(BytesIO(model_bytes))
        return artifact

    Modelo = load_artifact_from_gcs(model_path)
    encoder = load_artifact_from_gcs(encoder_path)

    Xtrain = Datos[Datos.columns[:-1]]

    Pred = Modelo.predict(Xtrain)
    Pred = encoder.inverse_transform(Pred)
    Pred = pd.DataFrame(Pred, columns=['Predicted'])
    
    #========================= Variables Auditoria ======================#
    user_id = client.query("SELECT SESSION_USER()").to_dataframe().iloc[0,0]
    start_time = generate_datetime_created()
    execution_date = generate_date_created()   

    print('=================== Tabla Final =====================')

    TablaFinal = pd.concat([Datos,Pred],axis=1)
    TablaFinal['CreationUser'] = user_id
    TablaFinal['ProcessDate'] = pd.to_datetime(execution_date, format='%Y-%m-%d')
    TablaFinal['LoadDate'] = pd.to_datetime(start_time)
    
    print(TablaFinal.head(5))
    
    pandas_gbq.to_gbq(TablaFinal, table_id, if_exists='append', project_id=project)

    print("Predicción generada y guardada en otro Proyecto de BigQuery.")

### Pipeline Principal

In [85]:
@kfp.dsl.pipeline(
    name="pipeline-classify-model", 
    description="intro",
    pipeline_root="gs://cloudstorage-mlops/Classification-Pipeline"
)

def main_pipeline(
    project: str,
    source_x_train_table: str,
    table_id: str,
    model_path: str,
    encoder_path: str,
    gcp_region: str = "us-central1",
):
    notification_email_task = VertexNotificationEmailOp(
        recipients = ["guy3hil@gmail.com"]
    )
    notification_email_task.set_display_name("NOTIFICATION_EMAIL")

    with dsl.ExitHandler(notification_email_task, name="Execute pipeline prediction"):

        validate_data_tables = validate_data(
            source_x_train_table = source_x_train_table
        )
        validate_data_tables.set_display_name("VALIDATE_COMPONENT")

        with dsl.Condition(
            validate_data_tables.outputs['condition']=="false",
            name="no-execute",
        ):
            error_op(msg="No se logro validar la creacion de la tabla")

        with dsl.Condition(
            validate_data_tables.outputs['condition']=="true",
            name="yes-execute",
        ):
            classify_task = classify(
                project = project,
                source_x_train_table = source_x_train_table,
                table_id = table_id,
                model_path = model_path,
                encoder_path = encoder_path
            )
            classify_task.set_display_name("CLASSIFICATION")

### Compilador

In [86]:
compiler.Compiler().compile(
    pipeline_func=main_pipeline,
    package_path="ClassifyPipeline-Proyecto.json"
)

### Funcionalidad auxiliar

In [87]:
# def upload_to_gcs(bucket_name, source_file_name, destination_blob_name):
#     storage_client = storage.Client()
#     bucket = storage_client.bucket(bucket_name)
#     blob = bucket.blob(destination_blob_name)
#     blob.upload_from_filename(source_file_name)
#     print(f"Archivo {source_file_name} subido a {destination_blob_name} en el bucket {bucket_name}.")


# bucket_name = "cloudstorage-mlops"
# destination_blob_name = "Prediction-Pipeline/prediction-pipeline.json"
# pipeline_file = "prediction-pipeline.json"
# upload_to_gcs(bucket_name, pipeline_file, destination_blob_name)

### Job en Vertex

In [88]:
aiplatform.init(project="proyecto-mlops-504619", location="us-central1")

job = aiplatform.PipelineJob(
    display_name="Pipeline de Clasificacion",
    template_path="ClassifyPipeline-Proyecto.json",
    enable_caching=False,
    project="proyecto-mlops-504619",
    location="us-central1",
    parameter_values={"project": "proyecto-mlops-504619", 
                      "source_x_train_table": "proyecto-fuente.BaseDatosIris.iris-test",
                      "table_id": "proyecto-produccion-505720.Resultados.predictions",
                      "model_path": "gs://cloudstorage-mlops/Training-Pipeline/Data/Model/XGBmodel.joblib",
                      "encoder_path": "gs://cloudstorage-mlops/Training-Pipeline/Data/Model/labelEncoder.joblib"
                     }
    #labels={"module": "ml", "application": "app", "chapter": "mlops", "company": "datapat", "environment": "dev", "owner": "xxxx"}
)


print('submit pipeline job ...')
job.submit(service_account="vertex-processing@proyecto-mlops-504619.iam.gserviceaccount.com")

submit pipeline job ...
Creating PipelineJob


PipelineJob created. Resource name: projects/884292398314/locations/us-central1/pipelineJobs/pipeline-classify-model-20260817083712
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/884292398314/locations/us-central1/pipelineJobs/pipeline-classify-model-20260817083712')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/pipeline-classify-model-20260817083712?project=884292398314
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/884292398314/locations/us-central1/pipelineJobs/pipeline-classify-model-20260817083712')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/pipeline-classify-model-20260817083712?project=884292398314
